# RAG Agent Evaluation

In this notebook, we'll evaluate the IT Help Desk RAG agent from Module 2 using comprehensive evaluation metrics. You'll learn how to measure agent quality using RAGAS metrics and LLM-as-a-judge techniques with NVIDIA models.

## Setup

First, let's install dependencies and load environment variables.

In [ ]:
%pip install -r ../requirements.txt > /dev/null
from dotenv import load_dotenv
_ = load_dotenv("../variables.env")
_ = load_dotenv("../secrets.env")

## Import Libraries

Import all the necessary libraries for evaluation:

In [ ]:
import json
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any

from evaluation_framework import (
    create_judge_llm,
    create_embeddings,
    evaluate_faithfulness,
    evaluate_relevancy,
    evaluate_helpfulness,
    evaluate_rag_response,
    calculate_aggregate_score
)

print("✅ Libraries imported successfully!")

## Load Test Dataset

We've prepared a comprehensive test dataset with IT help desk questions. Each test case includes:
- **Question**: The user's query
- **Ground Truth Answer**: The expected response  
- **Expected Context Keywords**: Keywords that should appear in retrieved documents
- **Category**: The type of question (password_management, network_access, etc.)

In [ ]:
# Load test cases
test_dataset_path = Path("../data/evaluation/rag_agent_test_cases.json")
with open(test_dataset_path, 'r') as f:
    test_dataset = json.load(f)

print(f"✅ Loaded {len(test_dataset)} test cases\n")
print("Example test case:")
print(f"Question: {test_dataset[0]['question']}")
print(f"Category: {test_dataset[0]['category']}")
print(f"Ground Truth: {test_dataset[0]['ground_truth'][:100]}...")

## Initialize the RAG Agent

Let's load the RAG agent we built in Module 2. This is the IT Help Desk agent that uses retrieval augmented generation.

In [ ]:
# Import the RAG agent
from rag_agent import AGENT, RETRIEVER

print("✅ RAG agent loaded successfully!")
print("Agent is ready to answer IT help desk questions.")

## Run Agent on Test Cases

Now let's run the agent on each test case and collect the results.

**Note**: This may take several minutes depending on the number of test cases. Each question requires the agent to:
1. Retrieve relevant contexts from the knowledge base
2. Generate a response using the LLM
3. Format the final answer

In [ ]:
results = []

print("Running agent on test cases...\n")
print("=" * 80)

for i, test_case in enumerate(test_dataset):
    print(f"\n[{i+1}/{len(test_dataset)}] Processing: {test_case['question'][:60]}...")
    
    try:
        # Run the agent
        response = AGENT.invoke({
            "messages": [{"role": "user", "content": test_case["question"]}]
        })
        
        # Extract the response
        agent_response = response["messages"][-1].content
        
        # Get retrieved contexts (for evaluation)
        # In a real scenario, you'd capture these from the agent's execution trace
        # For this notebook, we'll retrieve them directly to ensure accuracy
        retrieved_docs = RETRIEVER.invoke(test_case["question"])
        retrieved_contexts = [doc.page_content for doc in retrieved_docs]
        
        results.append({
            "question": test_case["question"],
            "agent_response": agent_response,
            "ground_truth": test_case["ground_truth"],
            "retrieved_contexts": retrieved_contexts,
            "category": test_case["category"]
        })
        
        print(f"✓ Response ({len(agent_response)} chars): {agent_response[:80]}...")
        print(f"✓ Retrieved {len(retrieved_contexts)} context chunks")
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        results.append({
            "question": test_case["question"],
            "agent_response": f"ERROR: {str(e)}",
            "ground_truth": test_case["ground_truth"],
            "retrieved_contexts": [],
            "category": test_case["category"]
        })

print("\n" + "=" * 80)
print(f"\n✅ Completed running agent on {len(results)} test cases!")

## Evaluate with LLM-as-a-Judge

Now let's evaluate each response using NVIDIA Nemotron models as judges. We'll assess three key qualities:

1. **Faithfulness**: Is the response grounded in the retrieved context?
2. **Relevancy**: Does the response address the user's question?
3. **Helpfulness**: Is the response actionable and useful?

This evaluation uses the evaluation framework we set up earlier.

In [ ]:
# Initialize judge model
print("Initializing NVIDIA Nemotron judge model...")
judge_llm = create_judge_llm()
print("✅ Judge model ready!\n")

print("Evaluating responses with LLM-as-a-judge...")
print("=" * 80)

# Evaluate each response
for i, result in enumerate(results):
    print(f"\n[{i+1}/{len(results)}] Evaluating: {result['question'][:60]}...")
    
    try:
        # Prepare context string
        context_str = "\n\n".join(result["retrieved_contexts"])
        
        # Run comprehensive evaluation
        eval_results = evaluate_rag_response(
            question=result["question"],
            response=result["agent_response"],
            context=context_str,
            judge_llm=judge_llm
        )
        
        # Store evaluation results (normalized to 0-1 scale)
        result["faithfulness_score"] = eval_results["faithfulness"].score / 5.0
        result["faithfulness_explanation"] = eval_results["faithfulness"].explanation
        result["relevancy_score"] = eval_results["relevancy"].score / 5.0
        result["relevancy_explanation"] = eval_results["relevancy"].explanation
        result["helpfulness_score"] = eval_results["helpfulness"].score / 5.0
        result["helpfulness_explanation"] = eval_results["helpfulness"].explanation
        
        # Calculate aggregate score
        result["aggregate_score"] = calculate_aggregate_score(eval_results)
        
        print(f"  Faithfulness: {result['faithfulness_score']:.2f} - {eval_results['faithfulness'].explanation[:50]}...")
        print(f"  Relevancy:    {result['relevancy_score']:.2f} - {eval_results['relevancy'].explanation[:50]}...")
        print(f"  Helpfulness:  {result['helpfulness_score']:.2f} - {eval_results['helpfulness'].explanation[:50]}...")
        print(f"  Overall:      {result['aggregate_score']:.2f}")
        
    except Exception as e:
        print(f"✗ Error during evaluation: {str(e)}")
        result["faithfulness_score"] = 0.0
        result["relevancy_score"] = 0.0
        result["helpfulness_score"] = 0.0
        result["aggregate_score"] = 0.0
        result["faithfulness_explanation"] = "Evaluation failed"
        result["relevancy_explanation"] = "Evaluation failed"
        result["helpfulness_explanation"] = "Evaluation failed"

print("\n" + "=" * 80)
print("\n✅ Evaluation complete!")

## Compute RAGAS Metrics

In addition to LLM-as-a-judge evaluation, let's compute RAGAS metrics. RAGAS provides specialized metrics for RAG systems:

- **Context Precision**: Measures if retrieved contexts are relevant (high precision = no irrelevant docs)
- **Context Recall**: Measures if all necessary info was retrieved (high recall = no missing info)
- **Faithfulness**: Measures if the answer is grounded in context
- **Answer Relevancy**: Measures if the answer addresses the question

**Note**: This section is optional and requires additional setup. Skip if you encounter issues.

In [ ]:
# Try to import and run RAGAS
try:
    from ragas import evaluate
    from ragas.metrics import (
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    )
    from datasets import Dataset
    
    print("Running RAGAS evaluation...\n")
    
    # Prepare data for RAGAS
    ragas_data = {
        "question": [r["question"] for r in results],
        "answer": [r["agent_response"] for r in results],
        "contexts": [r["retrieved_contexts"] for r in results],
        "ground_truth": [r["ground_truth"] for r in results],
    }
    
    ragas_dataset = Dataset.from_dict(ragas_data)
    
    # Create embeddings for RAGAS
    embeddings = create_embeddings()
    
    # Run RAGAS evaluation
    ragas_results = evaluate(
        dataset=ragas_dataset,
        metrics=[
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
        ],
        llm=judge_llm,
        embeddings=embeddings,
    )
    
    print("✅ RAGAS Metrics:")
    print(ragas_results)
    
except Exception as e:
    print(f"⚠️  RAGAS evaluation skipped: {str(e)}")
    print("This is optional - continuing with LLM-as-a-judge results.")

## Analyze Results

Let's analyze the evaluation results to understand agent performance across different dimensions.

In [ ]:
# Convert to DataFrame for easier analysis
df = pd.DataFrame(results)

print("=" * 80)
print("OVERALL EVALUATION RESULTS")
print("=" * 80)
print(f"\nMean Scores (0-1 scale):")
print(f"  Faithfulness:  {df['faithfulness_score'].mean():.3f}")
print(f"  Relevancy:     {df['relevancy_score'].mean():.3f}")
print(f"  Helpfulness:   {df['helpfulness_score'].mean():.3f}")
print(f"\nAggregate Score: {df['aggregate_score'].mean():.3f}")

print("\n" + "=" * 80)
print("SCORE DISTRIBUTION")
print("=" * 80)
print("\nFaithfulness:")
print(df['faithfulness_score'].describe())
print("\nRelevancy:")
print(df['relevancy_score'].describe())
print("\nHelpfulness:")
print(df['helpfulness_score'].describe())

## Performance by Category

Let's see how the agent performs across different categories of questions.

In [ ]:
print("=" * 80)
print("PERFORMANCE BY CATEGORY")
print("=" * 80)

category_stats = df.groupby('category')[[
    'faithfulness_score', 
    'relevancy_score', 
    'helpfulness_score',
    'aggregate_score'
]].mean()

print("\n", category_stats.round(3))

# Find best and worst performing categories
best_category = category_stats['aggregate_score'].idxmax()
worst_category = category_stats['aggregate_score'].idxmin()

print(f"\n✅ Best performing category: {best_category} ({category_stats.loc[best_category, 'aggregate_score']:.3f})")
print(f"⚠️  Worst performing category: {worst_category} ({category_stats.loc[worst_category, 'aggregate_score']:.3f})")

## Identify Problem Areas

Let's find specific questions where the agent performed poorly. These are opportunities for improvement.

In [ ]:
# Define threshold for "low" scores
threshold = 0.6  # Scores below 60% are concerning

print("=" * 80)
print(f"LOW-SCORING RESPONSES (Score < {threshold})")
print("=" * 80)

# Find responses with any score below threshold
low_scores = df[
    (df['faithfulness_score'] < threshold) |
    (df['relevancy_score'] < threshold) |
    (df['helpfulness_score'] < threshold)
]

if len(low_scores) > 0:
    print(f"\n⚠️  Found {len(low_scores)} responses with low scores:\n")
    
    for idx, row in low_scores.iterrows():
        print(f"\nQuestion: {row['question']}")
        print(f"Category: {row['category']}")
        print(f"Response: {row['agent_response'][:150]}...")
        print(f"\nScores:")
        print(f"  Faithfulness: {row['faithfulness_score']:.2f} - {row['faithfulness_explanation']}")
        print(f"  Relevancy:    {row['relevancy_score']:.2f} - {row['relevancy_explanation']}")
        print(f"  Helpfulness:  {row['helpfulness_score']:.2f} - {row['helpfulness_explanation']}")
        print("-" * 80)
else:
    print("\n✅ Excellent! All responses scored above threshold.")

## Custom Evaluation: Citation Quality

Let's add a custom evaluation to check if the agent properly cites sources with `[KB]` tags. This is important for IT help desk agents to show where information comes from.

In [ ]:
# Check citation quality
def has_citation(response: str) -> bool:
    """Check if response includes proper citations."""
    return "[KB]" in response

df['has_citation'] = df['agent_response'].apply(has_citation)
citation_rate = df['has_citation'].mean()

print("=" * 80)
print("CITATION QUALITY ANALYSIS")
print("=" * 80)
print(f"\nCitation Rate: {citation_rate:.1%}")
print(f"Responses with citations: {df['has_citation'].sum()}/{len(df)}")

if citation_rate < 0.8:
    print("\n⚠️  Warning: Citation rate is below 80%. Consider updating the system prompt.")
    print("\nResponses without citations:")
    no_citations = df[~df['has_citation']]
    for _, row in no_citations.iterrows():
        print(f"  - {row['question'][:60]}...")
else:
    print("\n✅ Good! Most responses include proper citations.")

## Custom Evaluation: Actionability

Let's evaluate whether responses provide clear, actionable steps. We'll use the judge model for this custom evaluation.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define custom evaluation prompt
custom_eval_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are evaluating IT help desk responses for actionability."),
    ("user", """
Evaluate this IT help desk response on actionability (1-5 scale):

Question: {question}
Response: {response}

Actionability criteria:
- Does it provide clear next steps?
- Are instructions specific and easy to follow?
- Does it help the user solve their problem?

Provide your score (1-5) and a brief explanation as JSON:
{{
  "score": <1-5>,
  "explanation": "<your explanation>"
}}
""")
])

custom_eval_chain = custom_eval_prompt | judge_llm

print("Running custom actionability evaluation...\n")

# Run custom evaluation on a sample
sample_size = min(5, len(results))
for i, result in enumerate(results[:sample_size]):
    print(f"[{i+1}/{sample_size}] Evaluating: {result['question'][:50]}...")
    
    try:
        custom_result = custom_eval_chain.invoke({
            "question": result["question"],
            "response": result["agent_response"]
        })
        
        # Try to parse JSON
        import re
        try:
            parsed = json.loads(custom_result.content)
            score = parsed.get("score", 0)
            explanation = parsed.get("explanation", "")
        except:
            # Fallback: extract score from text
            score_match = re.search(r'"score":\s*(\d+)', custom_result.content)
            score = int(score_match.group(1)) if score_match else 0
            explanation = "Could not parse full explanation"
        
        print(f"  Actionability: {score}/5 - {explanation[:60]}...\n")
        
    except Exception as e:
        print(f"  Error: {str(e)}\n")

print("✅ Custom evaluation complete!")

## Save Results

Let's save the evaluation results for future reference and tracking.

In [ ]:
# Ensure output directory exists
output_dir = Path("../data/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

# Save detailed results to CSV
results_path = output_dir / "rag_agent_results.csv"
df.to_csv(results_path, index=False)
print(f"✅ Detailed results saved to: {results_path}")

# Save summary statistics to JSON
summary = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "total_test_cases": len(df),
    "mean_scores": {
        "faithfulness": float(df['faithfulness_score'].mean()),
        "relevancy": float(df['relevancy_score'].mean()),
        "helpfulness": float(df['helpfulness_score'].mean()),
        "aggregate": float(df['aggregate_score'].mean())
    },
    "citation_rate": float(citation_rate),
    "low_scoring_count": len(low_scores),
    "categories": category_stats.to_dict()
}

summary_path = output_dir / "rag_agent_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Summary saved to: {summary_path}")

print("\n" + "=" * 80)
print("RESULTS SAVED")
print("=" * 80)

## Improvement Recommendations

Based on the evaluation results, let's generate specific recommendations for improving the agent.

In [ ]:
print("=" * 80)
print("IMPROVEMENT RECOMMENDATIONS")
print("=" * 80)

recommendations = []

# Check each metric
if df['faithfulness_score'].mean() < 0.8:
    recommendations.append(
        "📝 Faithfulness: Update system prompt to emphasize grounding responses in retrieved context. "
        "Consider adding explicit instructions to only use information from the knowledge base."
    )

if df['relevancy_score'].mean() < 0.8:
    recommendations.append(
        "🎯 Relevancy: Improve question understanding or add explicit relevance checking step. "
        "Consider adding few-shot examples of good vs. poor relevance."
    )

if df['helpfulness_score'].mean() < 0.8:
    recommendations.append(
        "💡 Helpfulness: Add more actionable steps and anticipate follow-up questions. "
        "Update prompts to include step-by-step instructions."
    )

if citation_rate < 0.8:
    recommendations.append(
        "📚 Citations: Strengthen citation requirements in system prompt. "
        "Make [KB] citations mandatory for all factual claims."
    )

if len(low_scores) > len(df) * 0.2:  # More than 20% low scores
    recommendations.append(
        "⚠️  Quality: More than 20% of responses scored below threshold. "
        "Consider reviewing retrieval parameters (chunk size, number of docs retrieved)."
    )

# Check category-specific issues
if worst_category and category_stats.loc[worst_category, 'aggregate_score'] < 0.7:
    recommendations.append(
        f"📂 Category-specific: '{worst_category}' questions perform poorly. "
        f"Consider adding more examples or improving knowledge base coverage for this category."
    )

if len(recommendations) == 0:
    print("\n🎉 Excellent! Agent is performing well across all metrics.")
    print("\nSuggestions for further improvement:")
    print("  - Test on more diverse or challenging cases")
    print("  - Add edge cases and adversarial examples")
    print("  - Monitor production performance for drift")
else:
    print("\nBased on evaluation results, here are specific recommendations:\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}\n")

print("=" * 80)

## Next Steps

Congratulations! You've successfully evaluated your RAG agent. Here's what you can do next:

1. **Implement Improvements**: Use the recommendations above to enhance your agent
2. **Re-evaluate**: Run this notebook again to measure the impact of changes
3. **Add More Test Cases**: Expand the test dataset with additional scenarios
4. **Monitor in Production**: Use these techniques to track agent performance over time
5. **Explore the Report Agent**: Try evaluating the report generation agent from Module 1

For more information, continue to the [Continuous Improvement](../.devx/3-agent-evaluation/continuous_improvement.md) lesson.